In [1]:
%cd ..

/Users/danorel/Workspace/Education/University/NYU/Research/xeda


In [2]:
import chromadb
import copy
import json
import typing as t
import uuid
import pathlib

from chromadb.utils import embedding_functions
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

from constants import (
    AWS_ACCESS_KEY_ID,
    AWS_SECRET_ACCESS_KEY,
    AWS_S3_ENDPOINT_URL,
    AWS_S3_REGION_NAME,
    AWS_S3_BUCKET_NAME,
    AWS_S3_USE_SSL,
    OPENAI_API_KEY,
    VECTOR_STORE_COLLECTION,
    VECTOR_STORE_HOST,
    VECTOR_STORE_PORT
)
from typings.pipeline import Pipeline
from pipeline.solid.pipeline_sampler import next_pipeline_iter
from utils.s3 import pull_keras_model

2024-04-05 11:44:37.885205: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
def node_to_encoding(node):
    annotation = node["annotation"]
    node_encoding = []
    for k, v in annotation.items():
        if isinstance(v, dict):
            for key in v:
                node_encoding.append(f"{k}_{key} = {v[key]}")
        else:
            node_encoding.append(f"{k} = {v}")
    return ', '.join(node_encoding)


def pipeline_to_splits(pipeline: Pipeline) -> t.List[Pipeline]:
    splits = []
    pipeline_encoding = []
    for node in reversed(pipeline):
        node_encoding = node_to_encoding(node)
        pipeline_encoding.append(node_encoding)
        splits.append(copy.deepcopy(pipeline_encoding))
    return splits


def pipeline_to_embedding(pipeline: Pipeline):
    pipeline_splits = pipeline_to_splits(pipeline)
    pipeline_payload = (
        [str(uuid.uuid4()) for _ in range(len(pipeline_splits))],
        [json.dumps(copy.deepcopy(pipeline)) for _ in range(len(pipeline_splits))],
        [';'.join(pipeline_split) for pipeline_split in pipeline_splits]
    )
    return pipeline_payload

In [4]:
pretrained_embeddings = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name="text-embedding-ada-002"
)

vector_store = chromadb.HttpClient(
    host=VECTOR_STORE_HOST, 
    port=VECTOR_STORE_PORT
)

In [5]:
vector_collection = vector_store.get_collection(VECTOR_STORE_COLLECTION)

In [6]:
root_dir = pathlib.Path.cwd()

In [7]:
annotated_dir, raw_dir = (
    root_dir / "data" / "eda4sum" / "annotated",
    root_dir / "data" / "eda4sum" / "raw"
)

In [8]:
annotated_pipelines = []
for annotated_file in annotated_dir.glob('*.json'):
    with annotated_file.open('r') as f:
        annotated_pipelines.append(json.load(f))

In [9]:
for annotated_pipeline in annotated_pipelines:
    (
        pipeline_ids,
        pipeline_documents,
        pipeline_sentences
    ) = pipeline_to_embedding(annotated_pipeline)
    vector_collection.add(
        ids=pipeline_ids,
        documents=pipeline_documents,
        embeddings=pretrained_embeddings(pipeline_sentences),
    )

In [10]:
pipeline_collection = vector_collection.get()

In [11]:
pipeline_collection.keys()

dict_keys(['ids', 'embeddings', 'metadatas', 'documents', 'data', 'uris'])

In [12]:
import s3fs

from dagster import EnvVar
from pipeline.resources import S3FSResource

fs = s3fs.S3FileSystem(
    key=AWS_ACCESS_KEY_ID,
    secret=AWS_SECRET_ACCESS_KEY,
    endpoint_url=AWS_S3_ENDPOINT_URL,
    use_ssl=AWS_S3_USE_SSL,
    client_kwargs={"region_name": AWS_S3_REGION_NAME},
)

# TODO: Change policy to universal / generalized one from the EDA UI
POLICY_NAME = "policy_0f2049f5-3807-4db4-9707-ea08a6d35383_concentrated"

In [13]:
!pip install --quiet scikit-learn

In [63]:
import random
import pandas as pd
import numpy as np

from tqdm import tqdm

from pipeline.solid.utils.model_manager import ModelManager
from pipeline.solid.utils.pipelines.pipeline_precalculated_sets import PipelineWithPrecalculatedSets
from sklearn.metrics.pairwise import cosine_similarity
from typings.pipeline import RequestData

def select_attributes(pipeline):
    return f"{[node['operator'] for node in pipeline]}"

def explore_pipeline(partial_pipeline: Pipeline, database_pipeline_cache, k: int):
    # This is a stub to simulate pipeline exploration.
    # In a real scenario, this would involve applying transformations or decisions based on an RL model.
    model_manager = ModelManager(database_pipeline_cache["galaxies"], models = {
        "set": pull_keras_model(
            s3fs=fs,
            bucket_name=AWS_S3_BUCKET_NAME,
            policy_name=POLICY_NAME,
            model_name="set_actor",
        ),
        "operation": pull_keras_model(
            s3fs=fs,
            bucket_name=AWS_S3_BUCKET_NAME,
            policy_name=POLICY_NAME,
            model_name="operation_actor",
        ),
        "set_op_counters": None,
    })

    partial_latest_node = partial_pipeline[-1]
    partial_latest_request_data = partial_latest_node.get("requestData")
    
    terminal_request_data = RequestData(**partial_latest_request_data)
    terminal_pipeline = partial_pipeline.copy()
    
    for i in range(len(partial_pipeline), len(partial_pipeline) + k):
        try:
            terminal_node, terminal_request_data = next_pipeline_iter(
                database_pipeline_cache,
                model_manager,
                terminal_request_data
            )
            terminal_pipeline.append(terminal_node)
        except ValueError:
            print(f"Unexpectedly exited from pipeline generation on step {i}. Saving pipeline as it is...")
            break
    
    return terminal_pipeline

def make_experiment(pipeline_index: int, database_pipeline_cache, type_of_similarity: str, k: int, verbose: bool = False):

    ids  = pipeline_collection['ids']
    docs = pipeline_collection['documents']

    partial_pipeline_id   = ids[pipeline_index]
    partial_pipeline = json.loads(docs[pipeline_index])
    
    rest_pipeline_ids = ids[:pipeline_index] + ids[pipeline_index+1:]
    rest_pipelines = [json.loads(doc) for doc in docs[:pipeline_index] + docs[pipeline_index+1:]]
    
    (
        partial_pipeline_ids,
        partial_pipeline_documents,
        partial_pipeline_sentences
    ) = pipeline_to_embedding(partial_pipeline)
    
    partial_annotation_embedding = pretrained_embeddings(partial_pipeline_sentences)
    partial_response = vector_collection.query(
        query_embeddings=partial_annotation_embedding,
        n_results=len(rest_pipeline_ids),
        include=["distances", "documents"]
    )
    
    terminal_pipeline = explore_pipeline(partial_pipeline, database_pipeline_cache, k=k)
    (
        terminal_pipeline_ids,
        terminal_pipeline_documents,
        terminal_pipeline_sentences
    ) = pipeline_to_embedding(partial_pipeline)
    terminal_annotation_embedding = pretrained_embeddings(terminal_pipeline_sentences)
    
    if verbose:
        print(f"Terminal pipeline: {select_attributes(partial_pipeline)}\n")

    partial_explanation_pipelines = partial_response['documents'][0]

    min_explanation_pipeline, max_explanation_pipeline = (
        json.loads(partial_explanation_pipelines[0]),
        json.loads(partial_explanation_pipelines[-1])
    )

    if verbose:
        print(f"Min pipeline: {min_explanation_distance}%, {select_attributes(min_explanation_pipeline)}")
        print(f"Max pipeline: {max_explanation_distance}%, {select_attributes(max_explanation_pipeline)}\n")

    (
        min_pipeline_ids,
        min_pipeline_documents,
        min_pipeline_sentences
    ) = pipeline_to_embedding(min_explanation_pipeline)
    min_annotation_embedding = pretrained_embeddings(min_pipeline_sentences)

    (
        max_pipeline_ids,
        max_pipeline_documents,
        max_pipeline_sentences
    ) = pipeline_to_embedding(max_explanation_pipeline)
    max_annotation_embedding = pretrained_embeddings(max_pipeline_sentences)

    terminal_annotation_embedding = np.array(terminal_annotation_embedding).mean(axis=0)
    min_annotation_embedding = np.array(min_annotation_embedding).mean(axis=0)
    max_annotation_embedding = np.array(max_annotation_embedding).mean(axis=0)
    
    if type_of_similarity == 'cosine':
        min_to_terminal_similarity = np.dot(terminal_annotation_embedding, min_annotation_embedding)/(np.linalg.norm(terminal_annotation_embedding)*np.linalg.norm(min_annotation_embedding))
        max_to_terminal_similarity = np.dot(terminal_annotation_embedding, min_annotation_embedding)/(np.linalg.norm(terminal_annotation_embedding)*np.linalg.norm(min_annotation_embedding))
    elif type_of_similarity == 'euclidian':
        min_to_terminal_similarity = np.linalg.norm(terminal_annotation_embedding - min_annotation_embedding)
        max_to_terminal_similarity = np.linalg.norm(terminal_annotation_embedding - max_annotation_embedding)
    elif type_of_similarity == 'manhattan':
        min_to_terminal_similarity = np.abs(terminal_annotation_embedding - min_annotation_embedding).sum()
        max_to_terminal_similarity = np.abs(terminal_annotation_embedding - max_annotation_embedding).sum()
    else:
        raise NotImplementedError("Unknown type of similarity")
        
    return {
        "partial_pipeline_id": partial_pipeline_id,
        "min_to_terminal_similarity": min_to_terminal_similarity,
        "max_to_terminal_similarity": max_to_terminal_similarity,
    }

def run_experiments(n: int = 1000, types_of_similarity: list = ['cosine'], exploration_steps: tuple = (3, 6)):
    measurements = pd.DataFrame()
    used_pipeline_ids = set()

    database_pipeline_cache = {}
    database_pipeline_cache["galaxies"] = PipelineWithPrecalculatedSets(
        "sdss",
        ["galaxies"],
        discrete_categories_count=10,
        min_set_size=10,
        exploration_columns=[
            "galaxies.u",
            "galaxies.g",
            "galaxies.r",
            "galaxies.i",
            "galaxies.z",
            "galaxies.petroRad_r",
            "galaxies.redshift",
        ],
        id_column="galaxies.objID",
    )

    for experiment in tqdm(range(n)):

        print(f"Starting '{experiment + 1}' experiment")

        for k in range(*exploration_steps):

            print(f"Starting a measurement with '{k}' exploration steps")

            for type_of_similarity in types_of_similarity:

                print(f"Starting a measurement with '{type_of_similarity}' type of similarity")
    
                trial = 0
                target_pipeline_index, target_pipeline_id = None, None
                while (target_pipeline_id not in used_pipeline_ids):
                    trial += 1
                    print(f"Making {trial} trial to sample partial pipeline within not seen pipelines")
                    (
                        target_pipeline_index,
                        target_pipeline_id
                    ) = random.sample(list(enumerate(pipeline_collection['ids'])), k=1)[0]
                    if target_pipeline_id is not None:
                        used_pipeline_ids.add(target_pipeline_id)
                
                measurements = pd.concat([
                    measurements,
                    pd.DataFrame([{
                        "experiment": experiment + 1,
                        "k": k,
                        "type_of_similarity": type_of_similarity,
                        **make_experiment(
                            target_pipeline_index, 
                            database_pipeline_cache, 
                            type_of_similarity,
                            k, 
                            verbose=False
                        )
                    }])
                ])
        
                print(f"Recored a measurement for {experiment + 1}-th sample {target_pipeline_id}")

    return measurements

In [ ]:
experiments_df = run_experiments(
    n=3,
    types_of_similarity=['cosine', 'euclidian'],
    exploration_steps=(3, 5),
)

  0%|                                                                                                                    | 0/3 [00:00<?, ?it/s]

Starting '1' experiment
Starting a measurement with '3' exploration steps
Starting a measurement with 'cosine' type of similarity
Making 1 trial to sample partial pipeline within not seen pipelines
1/1 [==============================] - 0s 292ms/step
[0.06409882 0.14747341 0.08064068 0.0659435  0.23150369 0.04980345
 0.09471154 0.06887679 0.0498749  0.14707322]
[0.10023842631825494, 0.2306205115889374, 0.12610676868909612, 0.10312315832549503, 0.36202796696875833, 0.07788316810945826, 0.0, 0.0, 0.0, 0.0]
1/1 [==============================] - 0s 299ms/step
[0.06279044 0.0711955  0.07162868 0.05015273 0.05730898 0.10387779
 0.06954382 0.05581452 0.04367214 0.05596778 0.07300258 0.06539377
 0.0573384  0.05875714 0.05047691 0.05307882]
[0.11458081206365597, 0.129918479212391, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.10213073819563424, 0.1332160571599578, 0.11933140985094015, 0.10463185579911677, 0.10722079323656893, 0.09211092846267238, 0.0968589260190628]
by_neighbors-&-galaxies.petroRad_r
1

 33%|███████████████████████████████████▋                                                                       | 1/3 [08:30<17:00, 510.43s/it]

Recored a measurement for 1-th sample 066dd55a-4d96-4fa0-a58f-67d10a6a5ea4
Starting '2' experiment
Starting a measurement with '3' exploration steps
Starting a measurement with 'cosine' type of similarity
Making 1 trial to sample partial pipeline within not seen pipelines


### Visualizations

In [ ]:
!pip install --quiet matplotlib seaborn

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
plot = experiments_df[['experiment', 'min_to_terminal_similarity', 'max_to_terminal_similarity']].plot(
    kind='bar', 
    x='experiment', 
    y=['min_to_terminal_similarity', 'max_to_terminal_similarity'],
    title='Max terminal similarity',
    figsize=(12, 5)
)
plt.show()

In [ ]:
experiments_success_condition = experiments_df['max_to_terminal_similarity'] > experiments_df['min_to_terminal_similarity']
experiments_success_amount = len(experiments_df.loc[experiments_success_condition].values)

In [ ]:
print(f'Success rate: {(experiments_success_amount / EXPERIMENTS) * 100}%')